# Screenshots (Hidden to Save Space)

![image-2](image-2.png)

![image-3](image-3.png)


![image-4](image-4.png)


![image-5](image-5.png)

![image-6](image-6.png)


![image-7](image-7.png)


![image-8](image-8.png)


![image-9](image-9.png)


**Denver Airbnb Analysis - Project Notes:**

- Calendar table grain: One record appears to represent one listing on one calendar date.
  
- Potential calendar key: (listing_id, date).
  
- Review table grain: One record appears to represent one individual review.
  
- Potential review key: id.
  
- Potential relationship: reviews.listing_id → listings.id.
  
- Important limitation: Calendar availability cannot automatically be interpreted as confirmed occupancy.

- id is Airbnb's unique identifier for a listing, while host_id identifies the host.

- neighbourhood_cleansed is calculated by geocoding each listing's latitude/longitude against public/open neighborhood boundaries, making it the better standardized geography field for our neighborhood analysis.

- price is the daily price in local currency, and the $ symbol is an export artifact. That will matter when we clean/cast the field for SQL analysis.

- availability_365 measures days marked available over the next 365 days, but Inside Airbnb explicitly warns that unavailable dates can mean either guest-booked or host-blocked. That confirms why we shouldn't call 365 - availability_365 "occupied nights."

- calculated_host_listings_count is specifically the number of that host's listings found in the current scrape within the city/region, which makes it more useful for our Denver market-concentration analysis than host_total_listings_count, whose calculation Airbnb does not disclose.

- reviews_per_month is an Inside Airbnb calculated field representing average monthly reviews over the listing's lifetime, and the dictionary even documents its approximate calculation.

--------

- There are 10 more distinct listing IDs represented in the calendar than there are listings in detailed_listings.csv.

- Every listing id has exactly 365 rows each in detailed_calendar

- Finding: 10 listing IDs represented in the detailed calendar are absent from the detailed listings snapshot.

- Referential integrity: The detailed reviews table contains 95 historical reviews associated with 8 listing IDs that are not present in the current listings snapshot. Additionally, 638 current listings have no records in the detailed reviews table.

- Issue: price is stored as text and contains currency symbols.
Treatment: Remove the currency formatting and convert the field to a numeric datatype before performing price calculations.

- Price is strongly right-skewed. A global IQR cutoff over-flags listings because room types have different price structures. Room-type-specific IQR is a better screening method, but flagged values should not be removed automatically because many may be legitimate high-end listings.

- Roughly 300 of the 4,251 priced listings are being flagged as statistical upper outliers. That’s about 7% of priced listings, which is plausible for a heavily right-skewed market.

- The biggest finding is the difference between the average ($276.04) and median ($176.45). The mean is much higher, which is strong evidence that the price distribution is right-skewed: a relatively small number of very expensive listings are pulling the average upward.

- From an analysis perspective, however, it's extremely important. Denver's Airbnb supply in this dataset is overwhelmingly entire homes/apartments: 85.71%. That tells us room_type should probably become one of our major segmentation dimensions throughout the project.

detailed_listings
- Grain: One row per listing
- Primary key candidate: id
- Uniqueness: Confirmed in current dataset
- NULLs in key: None

room_type and property_type
- 100% complete with four standardized categories. Entire home/apartment listings dominate the Denver dataset (85.71%), followed by private rooms (12.78%). No cleaning required. Retain as a key categorical segmentation variable.
- room_type answers "How much of the property does the guest get?", while property_type gives a more specific description of "What kind of accommodation/property is it?"
- "property_type contained 38 categories with substantial class imbalance. Inspection indicated that rare categories represented legitimate accommodation types rather than obvious data-quality errors. The original field was therefore retained, with broader analytical groupings considered for segment-level analysis."
- Entire home/apt - Entire home has the most listings at 1589 listings and 32.17% share. 
- I'm noticing there are private room types that could be better classified in 'Hotel room', including room in hotel, ...aparthotel, ...boutique hotel, ...hostel.
----------
- room_type: Low-cardinality broad classification with four complete, standardized categories. Useful for high-level market segmentation.
- property_type: Higher-cardinality field with 38 legitimate accommodation categories and a long-tailed distribution. Provides additional structural detail within each room type. Retain the raw field; broader derived categories may be created later if needed for analysis.
- room_type and property_type are complementary rather than redundant. room_type provides a broad classification of the rental arrangement, while property_type provides more granular information about the physical accommodation. Therefore, both fields should be retained.

Data-Cleaning for price 
- Issue: price is stored as text and contains currency symbols.
- Treatment: Remove the currency formatting and convert the field to a numeric datatype before performing price calculations. New column: cleaned_price

Listing counts for price thresholds
- $500–$999.99: 336 listings
- $1,000–$1,999.99: 53 listings
- $2,000+: 9 listings

Errors, Extreme Values, and Outliers 
- Data error — a value is demonstrably incorrect or unusable.
- Extreme value — a legitimate but unusually large/small observation.
- Statistical outlier — a value falling outside a defined statistical criterion such as 1.5×IQR.

------------

neighbourhood_cleansed
- no missing values
- 77 legitimate categories
- no apparent capitalization/whitespace duplication
- meaningful variation in listing counts
- strong analytical relevance
- compatible with our later geographic analysis
- neighbourhoods.csv contains 78 Denver neighborhoods. All 77 neighborhoods represented in the current listings snapshot match the reference table. Sun Valley is present in the neighborhood reference data but has no listings in the current detailed_listings.csv snapshot.

------------

accommodates
- accommodates contains no null or zero values and ranges from 1–16 guests. Listings at the upper bound appear consistent with large-group properties based on bedroom, bathroom, and listing-description information.
- There's another important consideration: 16 appears repeatedly. You may eventually discover that 16 is effectively a platform/category ceiling rather than the exact physical maximum occupancy of every property. For this project, however, you don't need to resolve that yet. Just don't interpret 16 as necessarily meaning "this property can accommodate exactly 16 and no more."

----------

bathrooms and bathrooms_text
- bathrooms_text reliably represents bathroom count and provides bathroom information for many listings where the numeric bathrooms field is missing. Among records where both fields are populated, the numeric values are consistent except for one identified discrepancy.
- When both fields are populated, bathrooms and the numeric bathroom count represented by bathrooms_text are highly consistent. One conflicting record was identified (bathrooms = 2, bathrooms_text = '3.5 baths'). This supports using bathrooms_text as a fallback source for missing numeric bathroom counts, subject to explicit handling of special text values.
- bathrooms_text is the more complete bathroom field. It contains usable bathroom information for 885 of the 887 records where numeric bathrooms is missing. Conversely, only 4 records have a numeric bathrooms value while bathrooms_text is missing. When both fields are populated, the values are highly consistent aside from one identified discrepancy.

----------

bedrooms/beds
- bedrooms and beds exhibit partially overlapping missingness. Approximately 75.6% of listings have both fields populated, while only 1.4% are missing both. Because each field is available for hundreds of records where the other is missing, both should be retained rather than treated as substitutes.
- One listing contains an unusually inconsistent capacity profile: 15 bedrooms, 50 beds, but an accommodates value of only 2. Because several capacity fields conflict, the record is flagged as a potential data-quality anomaly rather than automatically corrected or removed.
- For the eventual analysis, I would treat accommodates as your primary capacity variable, with bedrooms and beds as supplementary property characteristics. It's complete and directly represents the listing's stated guest capacity.

---------

minimum_nights
- Minimum-stay requirements are highly concentrated at a small number of values. One-night stays are the most common requirement (27.1%), while 29- and 30-night minimums collectively account for approximately 46.9% of listings. This suggests a substantial segment of the Denver Airbnb market is oriented toward longer stays. Extreme minimum-stay values also exist, reaching 500 nights, but these represent atypical listings and should not be interpreted as representative of the market.
- Minimum-stay policies show a highly polarized distribution. Approximately 46.9% of listings require 29–30 nights, while approximately 49.8% allow stays of six nights or fewer. Listings with minimum stays between 7 and 28 nights are comparatively rare at less than 1%.

----------

availability
- Inside Airbnb's available calendar status doesn't tell us why a date is unavailable. A host may have blocked dates, stopped accepting reservations temporarily, restricted the calendar, or the date may actually be booked.
- Calendar availability is substantially lower in the near term. Listings average approximately 30% availability over the next 30 days, compared with roughly 50–62% across subsequent future periods. This pattern may reflect a combination of reservations, host calendar restrictions, and other availability settings and should not be interpreted directly as occupancy.
- Availability fields were complete and remained within their expected bounds (0–30, 0–60, 0–90, and 0–365). Availability was substantially lower in the immediate 30-day window than in later future periods. Zero availability was particularly common over the next 30 days (35.1%), declining as the observation window expanded. Because unavailable calendar dates may represent bookings or host-imposed restrictions, availability is treated as a calendar-availability measure rather than an occupancy measure.
- The cumulative availability fields are logically consistent across the dataset. No listing has a shorter-window availability value greater than a longer-window availability value.

--------

- 638 listings have zero lifetime reviews, and exactly 638 listings have NULL reviews_per_month.
- reviews_per_month is NULL precisely for listings with no reviews rather than representing missing data in the conventional sense.
- For lifetime reviews, the middle 50% is roughly 3–85 reviews, yet the maximum reaches 1,798. Your earlier mean of 71.81 versus median of 21 reinforces the strong right skew.
- The LTM variable shows the same general phenomenon, although less extreme: 25% of listings received no reviews at all in the previous 12 months, while the upper 10% had at least roughly 38.
- This also tells us that when we eventually compare neighborhoods, median review activity may often be more representative than average review activity.
- Most Denver listings show some recent review activity, with approximately 72% recording at least one review during the previous 12 months. However, only about 43% recorded a review during the previous 30 days. Approximately 15% have historical reviews but none within the past year, while roughly 13% have never been reviewed.

number_of_reviews
- The mean being more than 3× the median, plus the 1,798 maximum, strongly suggests a right-skewed distribution. A relatively small number of highly reviewed listings are pulling the average upward.
- 638 listings have 0 lifetime reviews → about 12.9%

number_of_reviews_ltm
- That zero count is analytically interesting. A listing can have historical reviews but no recent reviews, which could indicate lower recent review activity, a listing that isn't currently operating much, or other factors. We shouldn't automatically call that "low demand."
- 1,365 listings have 0 reviews in the last 12 months → about 27.6%

number_of_reviews_l30d
- 2,803 listings have 0 reviews in the last 30 days → about 56.8%
- Finding: Recent review activity is highly concentrated. 56.8% of Denver listings recorded no reviews during the previous 30 days, resulting in a median of 0 despite a mean of 1.23 reviews per listing.

reviews_per_month
- is NULL precisely for listings with no reviews rather than representing missing data in the conventional sense.
- reviews_per_month contains 638 NULL values, all corresponding exactly to listings with zero lifetime reviews. These NULLs represent listings for which a monthly review rate has not been established rather than apparent data-quality errors.

---------

ratings
- Across all seven rating fields, you have 4,301 non-null values and 638 NULL values. That is especially notable because you previously found exactly 638 listings with number_of_reviews = 0.
- Your final query also found zero listings where number_of_reviews >= 1 while all seven review-score fields were NULL. So the evidence strongly suggests that the rating NULLs are structural: listings with no reviews do not have ratings.
- Ratings are extremely concentrated near 5.0. Every rating category has a median above 4.85, and most have Q3 = 5. This means these variables have a strong ceiling effect. A 4.7 rating might sound excellent in isolation, but within this particular market it could actually be relatively weak.
- Second, communication and check-in appear strongest. Their averages are 4.91 and 4.92 and medians are 4.98 and 4.97. Hosts generally seem to receive exceptionally high scores on these dimensions.
- Third, value stands out as the weakest dimension. Its mean of 4.79, median of 4.86 and Q1 of 4.74 are all noticeably below the other categories. That's potentially meaningful for our eventual pricing analysis: we may want to investigate whether higher-priced listings receive lower review_scores_value.
- Fourth, location is also somewhat lower than operational categories such as communication and check-in. That's interesting because location is largely a property characteristic rather than something a host can improve operationally.
- Listings with zero reviews have NULL review scores, while listings with at least one review have values for all seven review-score fields.
- Review-score fields contain 638 NULL values, corresponding exactly to listings with zero reviews. These NULLs represent unrated listings rather than erroneous missing data and will therefore be retained.

----------

host 
- Single-listing hosts make up 83.91% of all hosts, but account for only 56.37% of Denver's Airbnb listings.
- Hosts with more than 10 listings represent only 0.96% of hosts, yet control 15.02% of the listings in this dataset.
- 83.91% of hosts have exactly one listing, accounting for 56.37% of listings.
- 16.09% of hosts have multiple listings, accounting for 43.63% of listings.
- Hosts with more than 10 listings are only 0.96% of hosts, but account for 15.02% of listings.
- The 10 largest individual hosts account for 7.41% of all listings.
- Pricing: Superhost listings have a median asking price of $206 versus $135.82, roughly 52% higher. The average is also substantially higher ($323.77 vs. $212.46). Because both groups have averages substantially above their medians, that's another indication that price is right-skewed and reinforces our decision to pay attention to medians.
- Review activity: Superhost listings average 2.09 reviews/month versus 1.15 for non-Superhosts. That's a more useful comparison than the raw total-review numbers because there are different numbers of listings in each group.
- Large-host presence varies substantially across Denver neighborhoods. Some neighborhoods have relatively high shares of listings operated by hosts with portfolios exceeding 10 Denver listings.
- Validated calculated_host_listings_count against an independently calculated COUNT(*) by host_id. No discrepancies were identified among the 4,939 Denver listings, indicating that the field accurately represents the number of listings associated with each host within this dataset.

--------

estimated occupancy and revenue 
- Estimated Occupancy: Complete across all 4,939 listings, ranging from 0–255 estimated occupied nights. The median is 70 nights versus a mean of 103.91, indicating a right-skewed distribution. 1,365 listings (27.6%) have zero estimated occupancy. No negative or >365 values were identified.
- Estimated Revenue: Available for 4,251 listings and missing for 688. Median estimated annual revenue is $15,979 compared with a mean of $28,047.32, indicating substantial right skew from high-revenue listings. The maximum is $3,795,253.
- Revenue Nulls: All 688 missing estimated-revenue values correspond exactly to listings with missing prices. No listing with a populated price has missing estimated revenue.
- Metric Dependency: For all comparable records, estimated_revenue_l365d equals price × estimated_occupancy_l365d. Estimated revenue should therefore be treated as a derived estimate, not an independently observed measure of actual Airbnb revenue.

## detailed_calendar.csv

- Grain: one listing-calendar-date per row
- Composite key: (listing_id, date)
- detailed_calendar.csv grain and key: The dataset contains 1,806,385 rows representing 4,949 distinct listings. Each listing has exactly 365 records corresponding to 365 distinct dates. No duplicate (listing_id, date) combinations or NULL values in either key field were identified. Therefore, one row represents one listing on one calendar date, and (listing_id, date) is validated as the composite primary key.
- Cross-dataset integrity: detailed_calendar.csv contains 4,949 listing IDs compared with 4,939 listings in detailed_listings.csv, indicating 10 calendar listing IDs are absent from the current listings snapshot.
- 4,305 listings (87.0%) cover June 30, 2026 → June 29, 2027.
- 644 listings (13.0%) cover July 2, 2026 → July 1, 2027.
- Calendar date coverage: Each of the 4,949 listings contains exactly 365 unique calendar dates, but listings do not all share the same calendar window. 4,305 listings (87.0%) span June 30, 2026 through June 29, 2027, while 644 listings (13.0%) span July 2, 2026 through July 1, 2027. This suggests the calendar records were captured across different scrape dates/batches. Date-level analyses should account for differences in listing coverage near the boundaries of the dataset.

----------

availability
- Availability: The available field is complete and contains only the expected Boolean values. Across all 1,806,385 listing-date records, 1,021,669 (56.56%) are marked available and 784,716 (43.44%) are marked unavailable.
- False means the date is not available on the Airbnb calendar, but we don't know whether that's because it was booked, host-blocked, restricted, or unavailable for some other reason.
- Listing-level calendar availability: Availability varies substantially across Denver listings. The median listing is marked available for 236 of its 365 calendar dates (64.7%), compared with a mean of 206.44 days (56.6%). Approximately 40.9% of listings are available for at least 271 days, while 7.7% have zero available dates. These differences demonstrate substantial variation in calendar availability across listings. Because unavailable dates may represent either reservations or host-imposed restrictions, these values should not be interpreted directly as occupancy.
- Cross-dataset availability validation: Calendar-derived annual availability was compared with detailed_listings.availability_365 for all 4,939 listings represented in both datasets. 4,938 listings (99.98%) matched exactly. One listing was inconsistent: availability_365 reported 0 available days, while its 365 calendar records contained 207 available and 158 unavailable dates. The calendar record itself contained a complete 365-day window and plausible variation in availability over time. Because the source of the discrepancy cannot be determined from the available data, the record is retained and documented as a cross-dataset inconsistency rather than corrected.
- Calendar availability is substantially lower in the immediate future, rises considerably farther into the booking window, peaks around November–December in this snapshot, and then declines somewhat in later spring/summer months.

------------

minimum and maximum nights
- Minimum-stay requirements are considerably more dynamic than maximum-stay requirements. Approximately 24.5% of listings have multiple minimum-night requirements across their 365-day calendar, compared with only about 2.5% having multiple maximum-night requirements.
- That makes intuitive analytical sense as something worth investigating: hosts may configure different minimum stays for different dates. We don't need to determine why yet.
- Calendar stay requirements: minimum_nights and maximum_nights are complete across all 1,806,385 calendar records. Calendar-level minimum stays range from 1–500 nights, while maximum stays range from 1–1,125 nights. Stay requirements are not necessarily constant within a listing's calendar. Of 4,949 listings, 1,213 (24.5%) have multiple minimum-night requirements across their 365 dates, while 125 (2.5%) have multiple maximum-night requirements. Minimum-stay restrictions therefore exhibit substantially greater date-level variation than maximum-stay restrictions.
- The listing-level stay-restriction values are highly consistent with calendar-level restrictions, but calendar data contains additional date-specific variation that the listing-level fields do not capture.
- - Cross-dataset stay-restriction validation: Listing-level minimum_nights and maximum_nights were compared against all calendar-level values for each listing. Among 4,939 listings represented in both datasets, 4,938 (99.98%) had a listing-level minimum-night value occurring on at least one calendar date, and 4,938 (99.98%) had a listing-level maximum-night value occurring on at least one calendar date. Calendar-level restrictions can vary by date, particularly minimum stays, so listing-level fields should not be interpreted as necessarily applying uniformly across the entire 365-day calendar.
- Cross-dataset anomaly: One listing was identified as the sole exception across multiple listing-to-calendar validation tests. Its listing-level record reports minimum_nights = 0, maximum_nights = 0, and availability_365 = 0, while its calendar contains a complete 365-day window with minimum_nights = 1, maximum_nights = 365, and 207 available dates. The remaining 4,938 listings shared by both datasets reconcile successfully across these checks. Because the cause of the discrepancy cannot be established from the source data, the listing should be flagged as an anomalous cross-dataset record rather than automatically corrected.
- Stay-restriction consistency: 3,608 of 1,806,385 calendar records (~0.20%) have minimum_nights > maximum_nights. These records are highly concentrated, affecting only 14 of 4,949 listings (~0.28%). The condition appears logically inconsistent under a straightforward interpretation of minimum- and maximum-stay restrictions, but the records are not automatically classified as source errors because the underlying Airbnb rule semantics may be more complex. The affected listings were flagged for further inspection rather than corrected or removed.
- Calendar stay-rule consistency: 3,608 calendar records (~0.20%) have minimum_nights > maximum_nights, affecting 14 of 4,949 calendar listings (~0.28%). The condition is highly concentrated: 11 listings account for nearly all affected records, with 270–365 affected dates each, while three listings are affected on only one date. Most heavily affected listings have a constant 29-night minimum while their maximum-night restriction varies by date. Because the underlying Airbnb booking-rule semantics cannot be fully determined from the source data, these records are retained and flagged rather than automatically classified as errors or corrected.

## detailed_reviews.csv

- detailed_reviews grain/key: One row represents one individual Airbnb review. id is a valid primary-key candidate in the current dataset: all 354,741 review IDs are non-NULL and unique.
- Reviews/listings relationship: detailed_reviews.listing_id relates reviews to detailed_listings.id. Of 354,741 review records, 95 historical reviews reference 8 listing IDs that are absent from the current listings snapshot. Conversely, 638 current listings have no records in the detailed reviews table. The unmatched historical reviews are retained because the reviews dataset spans many years and may include listings no longer represented in the current snapshot.
- Review-count reconciliation: For all 4,939 current listings, detailed_listings.number_of_reviews exactly equals the number of corresponding records in detailed_reviews.csv. This includes the 638 listings reporting zero lifetime reviews, confirming that their absence from detailed_reviews is consistent with the listing-level review count rather than apparent missing review data.
- Reviewer-field completeness: All 354,741 review records contain both a reviewer_id and reviewer_name. No records contain a reviewer ID without a name or a name without an ID. Reviewer-level analysis should use reviewer_id rather than reviewer_name as the identifier.
- For every listing included by the INNER JOIN, the listing table's first_review and last_review fields exactly agree with the minimum and maximum dates observed in detailed_reviews.csv.

Data Dictionary: https://docs.google.com/spreadsheets/d/1iWCNJcSutYqpULSQHlNyGInUvHg2BoUGoNRIGa6Szc4/edit?usp=sharing

## reviews

In [2]:
/* number_of_reviews check */ 

SELECT 
	COUNT(*) AS total_rows,
	COUNT(number_of_reviews) AS non_null_rows,
	COUNT(*) - COUNT(number_of_reviews) AS null_rows,
	COUNT(CASE WHEN number_of_reviews = 0 THEN 1 END) AS zero_values,
	MIN(number_of_reviews) AS min_number_of_reviews,
	MAX(number_of_reviews) AS max_number_of_reviews,
	ROUND(AVG(number_of_reviews), 2) AS avg_number_of_reviews,
	MEDIAN(number_of_reviews) AS median_number_of_reviews
FROM 'detailed_listings.csv';

,total_rows,non_null_rows,null_rows,zero_values,min_number_of_reviews,max_number_of_reviews,avg_number_of_reviews,median_number_of_reviews
0,4939,4939,0,638,0,1798,71.81,21.0


In [4]:
/* number_of_reviews_ltm check */ 

SELECT 
	COUNT(*) AS total_rows,
	COUNT(number_of_reviews_ltm) AS non_null_rows,
	COUNT(*) - COUNT(number_of_reviews_ltm) AS null_rows,
	COUNT(CASE WHEN number_of_reviews_ltm = 0 THEN 1 END) AS zero_values,
	MIN(number_of_reviews_ltm) AS min_number_of_reviews_ltm,
	MAX(number_of_reviews_ltm) AS max_number_of_reviews_ltm,
	ROUND(AVG(number_of_reviews_ltm), 2) AS avg_number_of_reviews_ltm,
	MEDIAN(number_of_reviews_ltm) AS median_number_of_reviews_ltm
FROM 'detailed_listings.csv';

,total_rows,non_null_rows,null_rows,zero_values,min_number_of_reviews_ltm,max_number_of_reviews_ltm,avg_number_of_reviews_ltm,median_number_of_reviews_ltm
0,4939,4939,0,1365,0,224,12.72,3.0


In [5]:
/* number_of_reviews_l30d check */ 

SELECT 
	COUNT(*) AS total_rows,
	COUNT(number_of_reviews_l30d) AS non_null_rows,
	COUNT(*) - COUNT(number_of_reviews_l30d) AS null_rows,
	COUNT(CASE WHEN number_of_reviews_l30d = 0 THEN 1 END) AS zero_values,
	MIN(number_of_reviews_l30d) AS min_number_of_reviews_l30d,
	MAX(number_of_reviews_l30d) AS max_number_of_reviews_l30d,
	ROUND(AVG(number_of_reviews_l30d), 2) AS avg_number_of_reviews_l30d,
	MEDIAN(number_of_reviews_l30d) AS median_number_of_reviews_l30d
FROM 'detailed_listings.csv';

,total_rows,non_null_rows,null_rows,zero_values,min_number_of_reviews_l30d,max_number_of_reviews_l30d,avg_number_of_reviews_l30d,median_number_of_reviews_l30d
0,4939,4939,0,2803,0,18,1.23,0.0


In [6]:
/* reviews_per_month check */ 

SELECT 
	COUNT(*) AS total_rows,
	COUNT(reviews_per_month) AS non_null_rows,
	COUNT(*) - COUNT(reviews_per_month) AS null_rows,
	COUNT(CASE WHEN reviews_per_month = 0 THEN 1 END) AS zero_values,
	MIN(reviews_per_month) AS min_reviews_per_month,
	MAX(reviews_per_month) AS max_reviews_per_month,
	ROUND(AVG(reviews_per_month), 2) AS avg_reviews_per_month,
	MEDIAN(reviews_per_month) AS median_reviews_per_month
FROM 'detailed_listings.csv';

,total_rows,non_null_rows,null_rows,zero_values,min_reviews_per_month,max_reviews_per_month,avg_reviews_per_month,median_reviews_per_month
0,4939,4301,638,0,0.01,21.22,1.71,1.18


In [1]:
/* Checking consistency between fields */ 

SELECT  
	name, 
	description,
	price,
	neighbourhood_cleansed,
	property_type,
	room_type,
	number_of_reviews,
	number_of_reviews_ltm,
	number_of_reviews_l30d,
	reviews_per_month
FROM 'detailed_listings.csv'
WHERE 
	number_of_reviews_l30d > number_of_reviews_ltm
	OR number_of_reviews_ltm > number_of_reviews;

,name,description,price,neighbourhood_cleansed,property_type,room_type,number_of_reviews,number_of_reviews_ltm,number_of_reviews_l30d,reviews_per_month


In [3]:
/* Checking reviews_per_month for 0 number_of_reviews records */ 

SELECT  
	name, 
	description,
	price,
	neighbourhood_cleansed,
	property_type,
	room_type,
	reviews_per_month,
	number_of_reviews,
	number_of_reviews_ltm,
	number_of_reviews_l30d
FROM 'detailed_listings.csv'
WHERE 
	number_of_reviews = 0 AND reviews_per_month IS NOT NULL;

,name,description,price,neighbourhood_cleansed,property_type,room_type,reviews_per_month,number_of_reviews,number_of_reviews_ltm,number_of_reviews_l30d


In [4]:
/* Checking for zero values for reviews_per_month */ 

SELECT  
	name, 
	description,
	price,
	neighbourhood_cleansed,
	property_type,
	room_type,
	reviews_per_month,
	number_of_reviews,
	number_of_reviews_ltm,
	number_of_reviews_l30d
FROM 'detailed_listings.csv'
WHERE 
	reviews_per_month = 0;

,name,description,price,neighbourhood_cleansed,property_type,room_type,reviews_per_month,number_of_reviews,number_of_reviews_ltm,number_of_reviews_l30d


In [6]:
/* reviews_per_month null consistency test */ 

SELECT
    COUNT(*) AS inconsistent_rows
FROM 'detailed_listings.csv'
WHERE
    (number_of_reviews = 0 AND reviews_per_month IS NOT NULL)
    OR
    (number_of_reviews > 0 AND reviews_per_month IS NULL);

,inconsistent_rows
0,0


In [9]:
/* Investigating distribution for lifetime reviews */

SELECT
    QUANTILE_CONT(number_of_reviews, 0.25) AS q1,
    MEDIAN(number_of_reviews) AS median,
    QUANTILE_CONT(number_of_reviews, 0.75) AS q3,
	QUANTILE_CONT(number_of_reviews, 0.90) AS percentile_90
FROM 'detailed_listings.csv';

,q1,median,q3,percentile_90
0,3.0,21.0,85.0,199.0


In [10]:
/* Investigating distribution for LTM reviews */

SELECT
    QUANTILE_CONT(number_of_reviews_ltm, 0.25) AS q1,
    MEDIAN(number_of_reviews_ltm) AS median,
    QUANTILE_CONT(number_of_reviews_ltm, 0.75) AS q3,
	QUANTILE_CONT(number_of_reviews_ltm, 0.90) AS percentile_90
FROM 'detailed_listings.csv';

,q1,median,q3,percentile_90
0,0.0,3.0,20.0,38.0


In [16]:
/* How many listings fall into each of these categories?

Never reviewed
Lifetime reviews, but none in last 12 months
Reviews in last 12 months, but none in last 30 days
At least one review in last 30 days */

WITH categories AS (
	SELECT 
		CASE 
			WHEN number_of_reviews = 0 AND number_of_reviews_ltm = 0 AND number_of_reviews_l30d = 0
			THEN 'Never reviewed'
			WHEN number_of_reviews >= 1 AND number_of_reviews_ltm = 0 AND number_of_reviews_l30d = 0
			THEN 'Lifetime reviews, but none in last 12 months'
			WHEN number_of_reviews >= 1 AND number_of_reviews_ltm >= 1 AND number_of_reviews_l30d = 0
			THEN 'Reviews in last 12 months, but none in last 30 days'
			WHEN number_of_reviews >= 1 AND number_of_reviews_ltm >= 1 AND number_of_reviews_l30d >= 1
			THEN 'At least one review in last 30 days'
		END AS listings_review_categories
	FROM 'detailed_listings.csv'
),

total_listings AS (
	SELECT 
		COUNT(*) AS total
	FROM 'detailed_listings.csv'
),

categories_count AS (
	SELECT 
		listings_review_categories,
		COUNT(*) AS listings
	FROM categories 
	GROUP BY listings_review_categories
)

SELECT 
	c.listings_review_categories,
	c.listings,
	ROUND(c.listings/t.total * 100, 2) AS percent_of_total
FROM categories_count AS c
CROSS JOIN total_listings AS t
ORDER BY percent_of_total DESC;

,listings_review_categories,listings,percent_of_total
0,At least one review in last 30 days,2136,43.25
1,"Reviews in last 12 months, but none in last 30...",1438,29.12
2,"Lifetime reviews, but none in last 12 months",727,14.72
3,Never reviewed,638,12.92


## ratings

In [2]:
/* review_scores_rating */ 

SELECT 
	COUNT(*) AS total_rows,
	COUNT(review_scores_rating) AS non_null_rows,
	COUNT(*) - COUNT(review_scores_rating) AS null_rows,
	MIN(review_scores_rating) AS min_review_scores_rating,
	MAX(review_scores_rating) AS max_review_scores_rating,
	ROUND(AVG(review_scores_rating), 2) AS avg_review_scores_rating,
	QUANTILE_CONT(review_scores_rating, 0.25) AS q1_review_scores_rating,
	MEDIAN(review_scores_rating) AS median_review_scores_rating,
	QUANTILE_CONT(review_scores_rating, 0.75) AS q3_review_scores_rating
FROM 'detailed_listings.csv';

,total_rows,non_null_rows,null_rows,min_review_scores_rating,max_review_scores_rating,avg_review_scores_rating,q1_review_scores_rating,median_review_scores_rating,q3_review_scores_rating
0,4939,4301,638,1.0,5.0,4.87,4.85,4.94,5.0


In [3]:
/* review_scores_accuracy */ 

SELECT 
	COUNT(*) AS total_rows,
	COUNT(review_scores_accuracy) AS non_null_rows,
	COUNT(*) - COUNT(review_scores_accuracy) AS null_rows,
	MIN(review_scores_accuracy) AS min_review_scores_accuracy,
	MAX(review_scores_accuracy) AS max_review_scores_accuracy,
	ROUND(AVG(review_scores_accuracy), 2) AS avg_review_scores_accuracy,
	QUANTILE_CONT(review_scores_accuracy, 0.25) AS q1_review_scores_accuracy,
	MEDIAN(review_scores_accuracy) AS median_review_scores_accuracy,
	QUANTILE_CONT(review_scores_accuracy, 0.75) AS q3_review_scores_accuracy
FROM 'detailed_listings.csv';

,total_rows,non_null_rows,null_rows,min_review_scores_accuracy,max_review_scores_accuracy,avg_review_scores_accuracy,q1_review_scores_accuracy,median_review_scores_accuracy,q3_review_scores_accuracy
0,4939,4301,638,1.0,5.0,4.88,4.87,4.95,5.0


In [4]:
/* review_scores_cleanliness */ 

SELECT 
	COUNT(*) AS total_rows,
	COUNT(review_scores_cleanliness) AS non_null_rows,
	COUNT(*) - COUNT(review_scores_cleanliness) AS null_rows,
	MIN(review_scores_cleanliness) AS min_review_scores_cleanliness,
	MAX(review_scores_cleanliness) AS max_review_scores_cleanliness,
	ROUND(AVG(review_scores_cleanliness), 2) AS avg_review_scores_cleanliness,
	QUANTILE_CONT(review_scores_cleanliness, 0.25) AS q1_review_scores_cleanliness,
	MEDIAN(review_scores_cleanliness) AS median_review_scores_cleanliness,
	QUANTILE_CONT(review_scores_cleanliness, 0.75) AS q3_review_scores_cleanliness
FROM 'detailed_listings.csv';

,total_rows,non_null_rows,null_rows,min_review_scores_cleanliness,max_review_scores_cleanliness,avg_review_scores_cleanliness,q1_review_scores_cleanliness,median_review_scores_cleanliness,q3_review_scores_cleanliness
0,4939,4301,638,1.0,5.0,4.86,4.83,4.94,5.0


In [5]:
/* review_scores_checkin */ 

SELECT 
	COUNT(*) AS total_rows,
	COUNT(review_scores_checkin) AS non_null_rows,
	COUNT(*) - COUNT(review_scores_checkin) AS null_rows,
	MIN(review_scores_checkin) AS min_review_scores_checkin,
	MAX(review_scores_checkin) AS max_review_scores_checkin,
	ROUND(AVG(review_scores_checkin), 2) AS avg_review_scores_checkin,
	QUANTILE_CONT(review_scores_checkin, 0.25) AS q1_review_scores_checkin,
	MEDIAN(review_scores_checkin) AS median_review_scores_checkin,
	QUANTILE_CONT(review_scores_checkin, 0.75) AS q3_review_scores_checkin
FROM 'detailed_listings.csv';

,total_rows,non_null_rows,null_rows,min_review_scores_checkin,max_review_scores_checkin,avg_review_scores_checkin,q1_review_scores_checkin,median_review_scores_checkin,q3_review_scores_checkin
0,4939,4301,638,1.0,5.0,4.92,4.92,4.97,5.0


In [6]:
/* review_scores_communication */ 

SELECT 
	COUNT(*) AS total_rows,
	COUNT(review_scores_communication) AS non_null_rows,
	COUNT(*) - COUNT(review_scores_communication) AS null_rows,
	MIN(review_scores_communication) AS min_review_scores_communication,
	MAX(review_scores_communication) AS max_review_scores_communication,
	ROUND(AVG(review_scores_communication), 2) AS avg_review_scores_communication,
	QUANTILE_CONT(review_scores_communication, 0.25) AS q1_review_scores_communication,
	MEDIAN(review_scores_communication) AS median_review_scores_communication,
	QUANTILE_CONT(review_scores_communication, 0.75) AS q3_review_scores_communication
FROM 'detailed_listings.csv';

,total_rows,non_null_rows,null_rows,min_review_scores_communication,max_review_scores_communication,avg_review_scores_communication,q1_review_scores_communication,median_review_scores_communication,q3_review_scores_communication
0,4939,4301,638,1.0,5.0,4.91,4.92,4.98,5.0


In [7]:
/* review_scores_location */ 

SELECT 
	COUNT(*) AS total_rows,
	COUNT(review_scores_location) AS non_null_rows,
	COUNT(*) - COUNT(review_scores_location) AS null_rows,
	MIN(review_scores_location) AS min_review_scores_location,
	MAX(review_scores_location) AS max_review_scores_location,
	ROUND(AVG(review_scores_location), 2) AS avg_review_scores_location,
	QUANTILE_CONT(review_scores_location, 0.25) AS q1_review_scores_location,
	MEDIAN(review_scores_location) AS median_review_scores_location,
	QUANTILE_CONT(review_scores_location, 0.75) AS q3_review_scores_location
FROM 'detailed_listings.csv';

,total_rows,non_null_rows,null_rows,min_review_scores_location,max_review_scores_location,avg_review_scores_location,q1_review_scores_location,median_review_scores_location,q3_review_scores_location
0,4939,4301,638,1.0,5.0,4.85,4.82,4.92,5.0


In [8]:
/* review_scores_value */ 

SELECT 
	COUNT(*) AS total_rows,
	COUNT(review_scores_value) AS non_null_rows,
	COUNT(*) - COUNT(review_scores_value) AS null_rows,
	MIN(review_scores_value) AS min_review_scores_value,
	MAX(review_scores_value) AS max_review_scores_value,
	ROUND(AVG(review_scores_value), 2) AS avg_review_scores_value,
	QUANTILE_CONT(review_scores_value, 0.25) AS q1_review_scores_value,
	MEDIAN(review_scores_value) AS median_review_scores_value,
	QUANTILE_CONT(review_scores_value, 0.75) AS q3_review_scores_value
FROM 'detailed_listings.csv';

,total_rows,non_null_rows,null_rows,min_review_scores_value,max_review_scores_value,avg_review_scores_value,q1_review_scores_value,median_review_scores_value,q3_review_scores_value
0,4939,4301,638,1.0,5.0,4.79,4.74,4.86,4.94


In [11]:
SELECT 
	id, 
	name,
	description,
	neighbourhood_cleansed,
	number_of_reviews
FROM 'detailed_listings.csv'
WHERE 
	(number_of_reviews = 0 AND review_scores_rating IS NOT NULL) OR (number_of_reviews > 0 AND review_scores_rating IS NULL)
	OR (number_of_reviews = 0 AND review_scores_accuracy IS NOT NULL) OR (number_of_reviews > 0 AND review_scores_accuracy IS NULL)
	OR (number_of_reviews = 0 AND review_scores_cleanliness IS NOT NULL) OR (number_of_reviews > 0 AND review_scores_cleanliness IS NULL)
	OR (number_of_reviews = 0 AND review_scores_checkin IS NOT NULL) OR (number_of_reviews > 0 AND review_scores_checkin IS NULL)
	OR (number_of_reviews = 0 AND review_scores_communication IS NOT NULL) OR (number_of_reviews > 0 AND review_scores_communication IS NULL)
	OR (number_of_reviews = 0 AND review_scores_location IS NOT NULL) OR (number_of_reviews > 0 AND review_scores_location IS NULL)
	OR (number_of_reviews = 0 AND review_scores_value IS NOT NULL) OR (number_of_reviews > 0 AND review_scores_value IS NULL);

,id,name,description,neighbourhood_cleansed,number_of_reviews


## host

In [13]:
SELECT 
	host_id,
	host_since,
	host_is_superhost,
	host_response_rate,
	host_acceptance_rate,
	calculated_host_listings_count 
FROM 'detailed_listings.csv';

,host_id,host_since,host_is_superhost,host_response_rate,host_acceptance_rate,calculated_host_listings_count
0,666,None,True,None,None,1
1,783,None,False,None,None,1
2,933,None,True,None,None,1
3,990,None,False,None,None,1
4,2150,None,True,None,None,2
...,...,...,...,...,...,...
4934,569123788,None,True,None,None,14
4935,1717154169106481664,None,False,None,None,1
4936,22055594,None,True,None,None,2
4937,33706492,None,False,None,None,2


In [9]:
/* host_id */ 

SELECT 
	COUNT(*) AS total_rows,
	COUNT(DISTINCT host_id) AS distinct_hosts,
	COUNT(host_id) AS non_null_rows,
	COUNT(*) - COUNT(host_id) AS null_rows
FROM 'detailed_listings.csv';

,total_rows,distinct_hosts,non_null_rows,null_rows
0,4939,3318,4939,0


In [4]:
/* host_since */ 

SELECT *
FROM 'detailed_listings.csv'
WHERE host_since IS NOT NULL;

,id,listing_url,scrape_id,last_scraped,source,name,description,neighborhood_overview,picture_url,host_id,host_url,host_profile_id,host_profile_url,host_name,host_since,hosts_time_as_user_years,hosts_time_as_user_months,hosts_time_as_host_years,hosts_time_as_host_months,host_location,host_about,host_response_time,host_response_rate,host_acceptance_rate,host_is_superhost,host_thumbnail_url,host_picture_url,host_neighbourhood,host_listings_count,host_total_listings_count,host_verifications,host_has_profile_pic,host_identity_verified,neighbourhood,neighbourhood_cleansed,neighbourhood_group_cleansed,latitude,longitude,property_type,room_type,...,price_quote_price_per_night,price_quote_raw,minimum_nights,maximum_nights,minimum_minimum_nights,maximum_minimum_nights,minimum_maximum_nights,maximum_maximum_nights,minimum_nights_avg_ntm,maximum_nights_avg_ntm,calendar_updated,has_availability,availability_30,availability_60,availability_90,availability_365,calendar_last_scraped,number_of_reviews,number_of_reviews_ltm,number_of_reviews_l30d,availability_eoy,number_of_reviews_ly,estimated_occupancy_l365d,estimated_revenue_l365d,first_review,last_review,review_scores_rating,review_scores_accuracy,review_scores_cleanliness,review_scores_checkin,review_scores_communication,review_scores_location,review_scores_value,license,instant_bookable,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month


In [2]:
/* host_response_rate */ 

SELECT *
FROM 'detailed_listings.csv'
WHERE host_response_rate IS NOT NULL;

,id,listing_url,scrape_id,last_scraped,source,name,description,neighborhood_overview,picture_url,host_id,host_url,host_profile_id,host_profile_url,host_name,host_since,hosts_time_as_user_years,hosts_time_as_user_months,hosts_time_as_host_years,hosts_time_as_host_months,host_location,host_about,host_response_time,host_response_rate,host_acceptance_rate,host_is_superhost,host_thumbnail_url,host_picture_url,host_neighbourhood,host_listings_count,host_total_listings_count,host_verifications,host_has_profile_pic,host_identity_verified,neighbourhood,neighbourhood_cleansed,neighbourhood_group_cleansed,latitude,longitude,property_type,room_type,...,price_quote_price_per_night,price_quote_raw,minimum_nights,maximum_nights,minimum_minimum_nights,maximum_minimum_nights,minimum_maximum_nights,maximum_maximum_nights,minimum_nights_avg_ntm,maximum_nights_avg_ntm,calendar_updated,has_availability,availability_30,availability_60,availability_90,availability_365,calendar_last_scraped,number_of_reviews,number_of_reviews_ltm,number_of_reviews_l30d,availability_eoy,number_of_reviews_ly,estimated_occupancy_l365d,estimated_revenue_l365d,first_review,last_review,review_scores_rating,review_scores_accuracy,review_scores_cleanliness,review_scores_checkin,review_scores_communication,review_scores_location,review_scores_value,license,instant_bookable,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month


In [1]:
/* host_acceptance_rate */ 

SELECT *
FROM 'detailed_listings.csv'
WHERE host_acceptance_rate IS NOT NULL;

,id,listing_url,scrape_id,last_scraped,source,name,description,neighborhood_overview,picture_url,host_id,host_url,host_profile_id,host_profile_url,host_name,host_since,hosts_time_as_user_years,hosts_time_as_user_months,hosts_time_as_host_years,hosts_time_as_host_months,host_location,host_about,host_response_time,host_response_rate,host_acceptance_rate,host_is_superhost,host_thumbnail_url,host_picture_url,host_neighbourhood,host_listings_count,host_total_listings_count,host_verifications,host_has_profile_pic,host_identity_verified,neighbourhood,neighbourhood_cleansed,neighbourhood_group_cleansed,latitude,longitude,property_type,room_type,...,price_quote_price_per_night,price_quote_raw,minimum_nights,maximum_nights,minimum_minimum_nights,maximum_minimum_nights,minimum_maximum_nights,maximum_maximum_nights,minimum_nights_avg_ntm,maximum_nights_avg_ntm,calendar_updated,has_availability,availability_30,availability_60,availability_90,availability_365,calendar_last_scraped,number_of_reviews,number_of_reviews_ltm,number_of_reviews_l30d,availability_eoy,number_of_reviews_ly,estimated_occupancy_l365d,estimated_revenue_l365d,first_review,last_review,review_scores_rating,review_scores_accuracy,review_scores_cleanliness,review_scores_checkin,review_scores_communication,review_scores_location,review_scores_value,license,instant_bookable,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month


In [6]:
/* host_is_superhost */ 

SELECT 
	COUNT(*) AS total_rows,
	COUNT(host_is_superhost) AS non_null_values,
	COUNT(CASE WHEN host_is_superhost IS NULL THEN 1 END) AS null_values,
	COUNT(CASE WHEN host_is_superhost IS TRUE THEN 1 END) AS true_values,
	COUNT(CASE WHEN host_is_superhost IS FALSE THEN 1 END) AS false_values
FROM 'detailed_listings.csv';

,total_rows,non_null_values,null_values,true_values,false_values
0,4939,4939,0,2700,2239


In [10]:
/* count of listings for each host */ 

SELECT 
	host_id,
	COUNT(*) AS listings
FROM 'detailed_listings.csv'
GROUP BY host_id
ORDER BY listings DESC;

,host_id,listings
0,121718072,69
1,8672659,55
2,748452521,40
3,142202943,31
4,435252208,31
...,...,...
3313,505564080,1
3314,15863508,1
3315,121073099,1
3316,155106984,1


In [13]:
/* host listings category */ 

WITH host_count AS (
	SELECT 
		host_id,
		COUNT(*) AS listings
	FROM 'detailed_listings.csv'
	GROUP BY host_id
),
	
host_category AS (
	SELECT 
		host_id, 
		listings,
		CASE 
			WHEN listings = 1 THEN '1 Listing'
			WHEN listings = 2 THEN '2 Listings'
			WHEN listings BETWEEN 3 AND 5 THEN '3-5 Listings'
			WHEN listings BETWEEN 6 AND 10 THEN '6-10 Listings'
			ELSE 'Over 10 Listings'
		END AS host_listings_category
	FROM host_count	
),

host_category_calc AS (
	SELECT 
		host_listings_category,
		COUNT(*) AS host_count,
		SUM(listings) AS listing_count
	FROM host_category 
	GROUP BY host_listings_category
),

totals AS (
	SELECT 
		COUNT(*) AS total_listings,
		COUNT(DISTINCT host_id) AS total_hosts
	FROM 'detailed_listings.csv'
)

SELECT 
	h.host_listings_category,
	h.host_count,
	ROUND(h.host_count/t.total_hosts * 100, 2) AS pct_of_hosts,
	h.listing_count,
	ROUND(h.listing_count/t.total_listings * 100, 2) AS pct_of_listings
FROM host_category_calc AS h
CROSS JOIN totals AS t
ORDER BY h.host_count DESC;

,host_listings_category,host_count,pct_of_hosts,listing_count,pct_of_listings
0,1 Listing,2784,83.91,2784.0,56.37
1,2 Listings,325,9.80,650.0,13.16
2,3-5 Listings,136,4.10,481.0,9.74
3,6-10 Listings,41,1.24,282.0,5.71
4,Over 10 Listings,32,0.96,742.0,15.02


In [2]:
/* Top 10 hosts by Denver listing count */

WITH host_count AS (
    SELECT
        host_id,
        COUNT(*) AS listings
    FROM 'detailed_listings.csv'
    GROUP BY host_id
),

host_rank AS (
    SELECT
        host_id,
        listings,
        ROW_NUMBER() OVER (ORDER BY listings DESC) AS host_rank,
        SUM(listings) OVER () AS total_listings
    FROM host_count
)

SELECT
    host_rank,
    host_id,
    listings,
    ROUND(listings / total_listings * 100, 2) AS pct_of_all_listings,
FROM host_rank
WHERE host_rank <= 10
ORDER BY host_rank;

,host_rank,host_id,listings,pct_of_all_listings
0,1,121718072,69,1.40
1,2,8672659,55,1.11
2,3,748452521,40,0.81
3,4,142202943,31,0.63
4,5,435252208,31,0.63
5,6,460083844,30,0.61
6,7,6658113,29,0.59
7,8,24326234,28,0.57
8,9,14126753,27,0.55
9,10,34955862,26,0.53


In [3]:
/* What percentage of Denver's listings are controlled by the 10 largest hosts? (Collectively) */ 

WITH host_count AS (
    SELECT
        host_id,
        COUNT(*) AS listings
    FROM 'detailed_listings.csv'
    GROUP BY host_id
),

host_rank AS (
    SELECT
        host_id,
        listings,
        ROW_NUMBER() OVER (ORDER BY listings DESC) AS host_rank,
        SUM(listings) OVER () AS total_listings
    FROM host_count
)

SELECT
    COUNT(*) AS top_hosts,
    SUM(listings) AS top_host_listings,
    ROUND(
        SUM(listings) / MAX(total_listings) * 100,
        2
    ) AS pct_of_all_listings
FROM host_rank
WHERE host_rank <= 10;

,top_hosts,top_host_listings,pct_of_all_listings
0,10,366.0,7.41


In [7]:
/* Top 10 host cumulative concentration */ 

WITH host_count AS (
    SELECT
        host_id,
        COUNT(*) AS listings
    FROM 'detailed_listings.csv'
    GROUP BY host_id
),

host_rank AS (
    SELECT
        host_id,
		ROW_NUMBER() OVER (ORDER BY listings DESC, host_id) AS host_rank,
        listings,
		SUM(listings) OVER (
			ORDER BY listings DESC, host_id
			ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
		) AS cumulative_listings,
        SUM(listings) OVER () AS total_listings
    FROM host_count
)

SELECT
    host_rank,
    host_id,
    listings,
	cumulative_listings,
    ROUND(listings / total_listings * 100, 2) AS pct_of_all_listings,
	ROUND(cumulative_listings / total_listings * 100, 2) AS cumulative_pct
FROM host_rank
WHERE host_rank <= 10
ORDER BY host_rank;

,host_rank,host_id,listings,cumulative_listings,pct_of_all_listings,cumulative_pct
0,1,121718072,69,69.0,1.40,1.40
1,2,8672659,55,124.0,1.11,2.51
2,3,748452521,40,164.0,0.81,3.32
3,4,142202943,31,195.0,0.63,3.95
4,5,435252208,31,226.0,0.63,4.58
5,6,460083844,30,256.0,0.61,5.18
6,7,6658113,29,285.0,0.59,5.77
7,8,24326234,28,313.0,0.57,6.34
8,9,14126753,27,340.0,0.55,6.88
9,10,34955862,26,366.0,0.53,7.41


In [3]:
/* How do you determine whether each host falls within the top 1%, 5%, or 10% of the 3,318 hosts? */ 

WITH host_count AS (
    SELECT
        host_id,
        COUNT(*) AS listings
    FROM 'detailed_listings.csv'
    GROUP BY host_id
),

host_rank AS (
    SELECT
        host_id,
        listings,
        ROW_NUMBER() OVER (
            ORDER BY listings DESC, host_id
        ) AS host_rank,
        COUNT(*) OVER () AS total_hosts,
    FROM host_count
),

host_percentile AS (
	SELECT 
		host_id,
		listings,
		CASE 
			WHEN host_rank * 1.0 / total_hosts <= 0.01 THEN 'Top 1%'
			WHEN host_rank * 1.0 / total_hosts > 0.01 AND host_rank * 1.0 / total_hosts <= 0.05 THEN '1-5%'
			WHEN host_rank * 1.0 / total_hosts > 0.05 AND host_rank * 1.0 / total_hosts <= 0.1 THEN '5-10%'
		ELSE 'Other Hosts'
		END AS host_group
	FROM host_rank
),

host_group AS (
	SELECT 
		host_group,
		COUNT(*) AS host_count,
		SUM(listings) AS listing_count,
		SUM(SUM(listings)) OVER () AS total_listings
	FROM host_percentile
	GROUP BY host_group
)

SELECT 
	host_group,
	host_count,
	listing_count,
	ROUND(listing_count / total_listings * 100, 2) AS pct_of_all_listings
FROM host_group
ORDER BY host_count;

,host_group,host_count,listing_count,pct_of_all_listings
0,Top 1%,33,752.0,15.23
1,1-5%,132,621.0,12.57
2,5-10%,166,376.0,7.61
3,Other Hosts,2987,3190.0,64.59


In [9]:
/* Superhost analysis */ 

WITH cleaned_prices AS (
	SELECT 
		host_is_superhost, number_of_reviews, number_of_reviews_l30d, reviews_per_month, review_scores_rating, availability_365,	
		CAST (
			REPLACE(
				REPLACE(price, '$', ''),
				',',
				''
			)
			AS DECIMAL(10, 2)
		) AS cleaned_price
	FROM 'detailed_listings.csv'
) 


SELECT 
	host_is_superhost,
	COUNT(*) AS listings,
	MEDIAN(cleaned_price) AS median_price,
	ROUND(AVG(cleaned_price), 2) AS avg_price,
	SUM(number_of_reviews) AS total_reviews,
	SUM(number_of_reviews_l30d) AS total_reviews_l30d,
	ROUND(AVG(reviews_per_month), 2) AS avg_reviews_per_month,
	ROUND(AVG(review_scores_rating), 2) AS avg_review_scores_rating,
	ROUND(AVG(availability_365), 2) AS avg_availability_365
FROM cleaned_prices
GROUP BY host_is_superhost;

,host_is_superhost,listings,median_price,avg_price,total_reviews,total_reviews_l30d,avg_reviews_per_month,avg_review_scores_rating,avg_availability_365
0,False,2239,135.82,212.48,85265.0,1406.0,1.15,4.81,207.91
1,True,2700,206.00,323.77,269381.0,4675.0,2.09,4.91,204.74


In [13]:
/* Geographic concentration of larger hosts */ 

WITH host_count AS (
	SELECT 
		host_id,
		COUNT(*) AS host_listings
	FROM 'detailed_listings.csv'
	GROUP BY host_id 
),

host_category AS (
	SELECT 
		l.id,
		l.neighbourhood_cleansed,
		h.host_id,
		h.host_listings,
		CASE 
			WHEN h.host_listings = 1 THEN '1 Listing'
			WHEN h.host_listings = 2 THEN '2 Listings'
			WHEN h.host_listings BETWEEN 3 AND 5 THEN '3-5 Listings'
			WHEN h.host_listings BETWEEN 6 AND 10 THEN '6-10 Listings'
			ELSE 'Over 10 Listings'
		END AS host_size
	FROM 'detailed_listings.csv' AS l
	INNER JOIN host_count AS h
		ON l.host_id = h.host_id
),

neighbourhood_host_size AS (
	SELECT 
		neighbourhood_cleansed,
		COUNT(CASE WHEN host_size = 'Over 10 Listings' THEN 1 END) AS over_10_host_listings,
		COUNT(*) AS total_neighbourhood_listings
	FROM host_category
	GROUP BY neighbourhood_cleansed
)

SELECT 
	neighbourhood_cleansed,
	total_neighbourhood_listings,
	over_10_host_listings,
	ROUND(over_10_host_listings/total_neighbourhood_listings * 100, 2) AS large_host_pct
FROM neighbourhood_host_size
ORDER BY large_host_pct DESC;

,neighbourhood_cleansed,total_neighbourhood_listings,over_10_host_listings,large_host_pct
0,Civic Center,42,27,64.29
1,City Park,59,32,54.24
2,Indian Creek,9,4,44.44
3,Hale,91,40,43.96
4,CBD,165,61,36.97
...,...,...,...,...
72,Southmoor Park,4,0,0.00
73,Country Club,12,0,0.00
74,University,33,0,0.00
75,Hampden,26,0,0.00


In [16]:
WITH host_listings AS (
	SELECT 
		host_id,
		COUNT(*) AS listings,
	FROM 'detailed_listings.csv'
	GROUP BY host_id
)

SELECT 
	l.host_id,
	l.calculated_host_listings_count,
	h.listings,
FROM 'detailed_listings.csv' AS l
LEFT JOIN host_listings AS h
	ON l.host_id = h.host_id
WHERE h.host_id IS NULL OR l.calculated_host_listings_count != h.listings;

,host_id,calculated_host_listings_count,listings


## Estimated occupancy and revenue

In [17]:
/* Estimated occupancy profile */ 

SELECT
    COUNT(*) AS total_rows,
    COUNT(estimated_occupancy_l365d) AS non_null_rows,
    COUNT(*) - COUNT(estimated_occupancy_l365d) AS null_rows,
    COUNT(CASE WHEN estimated_occupancy_l365d = 0 THEN 1 END) AS zero_values,
    MIN(estimated_occupancy_l365d) AS min_occupancy,
    MAX(estimated_occupancy_l365d) AS max_occupancy,
    ROUND(AVG(estimated_occupancy_l365d), 2) AS avg_occupancy,
    MEDIAN(estimated_occupancy_l365d) AS median_occupancy,
    QUANTILE_CONT(estimated_occupancy_l365d, 0.25) AS q1,
    QUANTILE_CONT(estimated_occupancy_l365d, 0.75) AS q3
FROM 'detailed_listings.csv';

,total_rows,non_null_rows,null_rows,zero_values,min_occupancy,max_occupancy,avg_occupancy,median_occupancy,q1,q3
0,4939,4939,0,1365,0,255,103.91,70.0,0.0,189.0


In [18]:
/* Estimated revenue profile */ 

SELECT
    COUNT(*) AS total_rows,
    COUNT(estimated_revenue_l365d) AS non_null_rows,
    COUNT(*) - COUNT(estimated_revenue_l365d) AS null_rows,
    COUNT(CASE WHEN estimated_revenue_l365d = 0 THEN 1 END) AS zero_values,
    MIN(estimated_revenue_l365d) AS min_revenue,
    MAX(estimated_revenue_l365d) AS max_revenue,
    ROUND(AVG(estimated_revenue_l365d), 2) AS avg_revenue,
    MEDIAN(estimated_revenue_l365d) AS median_revenue,
    QUANTILE_CONT(estimated_revenue_l365d, 0.25) AS q1,
    QUANTILE_CONT(estimated_revenue_l365d, 0.75) AS q3
FROM 'detailed_listings.csv';

,total_rows,non_null_rows,null_rows,zero_values,min_revenue,max_revenue,avg_revenue,median_revenue,q1,q3
0,4939,4251,688,1012,0,3795253,28047.32,15979.0,1636.5,40310.5


In [19]:
/* Investigating extreme revenue values */ 

SELECT
    id,
    neighbourhood_cleansed,
    room_type,
    price,
    estimated_occupancy_l365d,
    estimated_revenue_l365d
FROM 'detailed_listings.csv'
WHERE estimated_revenue_l365d IS NOT NULL
ORDER BY estimated_revenue_l365d DESC
LIMIT 20;

,id,neighbourhood_cleansed,room_type,price,estimated_occupancy_l365d,estimated_revenue_l365d
0,35382615,Stapleton,Entire home/apt,"$23,573.00",161,3795253
1,1119495367723110346,Union Station,Entire home/apt,"$1,549.00",255,394995
2,45302598,Five Points,Entire home/apt,"$1,703.00",231,393393
3,919426282565454215,Sloan Lake,Entire home/apt,"$1,377.00",255,351135
4,985655038490529659,Highland,Entire home/apt,"$1,275.00",255,325125
5,1240902940175353325,Ruby Hill,Entire home/apt,"$1,252.00",245,306740
6,1027460321611540044,University Hills,Entire home/apt,"$1,116.52",255,284713
7,1281379734763991140,Sunnyside,Entire home/apt,"$1,075.00",255,274125
8,18753893,Highland,Entire home/apt,"$1,100.00",238,261800
9,1365644975955745566,Sloan Lake,Entire home/apt,"$1,051.00",231,242781


In [23]:
/* Checking for impossible values */

SELECT 
	id,
    neighbourhood_cleansed,
    room_type,
    price,
    estimated_occupancy_l365d,
    estimated_revenue_l365d
FROM 'detailed_listings.csv'
WHERE estimated_occupancy_l365d < 0
   OR estimated_occupancy_l365d > 365;

,id,neighbourhood_cleansed,room_type,price,estimated_occupancy_l365d,estimated_revenue_l365d


In [24]:
/* Check whether occupancy and revenue nulls behave consistently */

SELECT 
	id,
    neighbourhood_cleansed,
    room_type,
    price,
    estimated_occupancy_l365d,
    estimated_revenue_l365d
FROM 'detailed_listings.csv'
WHERE 
	(estimated_occupancy_l365d IS NULL AND estimated_revenue_l365d IS NOT NULL)
	OR (estimated_occupancy_l365d IS NOT NULL AND estimated_revenue_l365d IS NULL);

,id,neighbourhood_cleansed,room_type,price,estimated_occupancy_l365d,estimated_revenue_l365d
0,2235726,Union Station,Entire home/apt,None,0,<NA>
1,2267500,West Highland,Entire home/apt,None,180,<NA>
2,2875696,Windsor,Entire home/apt,None,0,<NA>
3,3138055,Capitol Hill,Entire home/apt,None,0,<NA>
4,3280778,Athmar Park,Entire home/apt,None,0,<NA>
...,...,...,...,...,...,...
683,1689993897520995417,Athmar Park,Private room,None,0,<NA>
684,1665595562861159962,Goldsmith,Entire home/apt,None,0,<NA>
685,1697434287376782456,Athmar Park,Entire home/apt,None,0,<NA>
686,1668334949776516711,Lincoln Park,Entire home/apt,None,0,<NA>


In [26]:
SELECT 
	COUNT(CASE WHEN estimated_revenue_l365d IS NULL AND price IS NULL THEN 1 END) AS both_null,
	COUNT(CASE WHEN estimated_revenue_l365d IS NULL AND price IS NOT NULL THEN 1 END) AS price_not_null
FROM 'detailed_listings.csv';

,both_null,price_not_null
0,688,0


In [28]:
WITH cleaned_prices AS (
	SELECT 
		estimated_revenue_l365d, estimated_occupancy_l365d,	
		CAST (
			REPLACE(
				REPLACE(price, '$', ''),
				',',
				''
			)
			AS DECIMAL(10, 2)
		) AS cleaned_price
	FROM 'detailed_listings.csv'
) 

SELECT 
	estimated_revenue_l365d,
	ROUND(cleaned_price * estimated_occupancy_l365d) AS estimated_revenue_formula_check
FROM cleaned_prices
WHERE estimated_revenue_l365d != ROUND(cleaned_price * estimated_occupancy_l365d);

,estimated_revenue_l365d,estimated_revenue_formula_check


# detailed_calendar.csv profiling

## Profiling the dataset structure

In [32]:
SELECT
    COUNT(*) AS row_count,
	4949 * 365 AS listings_calc_check, -- going off the 4949 distinct listings in calendar, not listings
    COUNT(DISTINCT listing_id) AS unique_listings,
    MIN(date) AS earliest_date,
    MAX(date) AS latest_date
FROM 'detailed_calendar.csv';

,row_count,listings_calc_check,unique_listings,earliest_date,latest_date
0,1806385,1806385,4949,2026-06-30 00:00:00+00:00,2027-07-01 00:00:00+00:00


In [2]:
SELECT
    listing_id,
    COUNT(*) AS row_count,
    COUNT(DISTINCT date) AS unique_dates
FROM 'detailed_calendar.csv'
GROUP BY listing_id
HAVING COUNT(*) != 365 OR COUNT(DISTINCT date) != 365
ORDER BY row_count;

,listing_id,row_count,unique_dates


In [3]:
SELECT
    listing_id,
    date,
    COUNT(*) AS row_count
FROM 'detailed_calendar.csv'
GROUP BY
    listing_id,
    date
HAVING COUNT(*) > 1;

,listing_id,date,row_count


In [4]:
SELECT
    SUM(CASE WHEN listing_id IS NULL THEN 1 ELSE 0 END) AS null_listing_ids,
    SUM(CASE WHEN date IS NULL THEN 1 ELSE 0 END) AS null_dates
FROM 'detailed_calendar.csv';

,null_listing_ids,null_dates
0,0.0,0.0


In [5]:
WITH listing_ranges AS (
    SELECT
        listing_id,
        MIN(date) AS start_date,
        MAX(date) AS end_date
    FROM 'detailed_calendar.csv'
    GROUP BY listing_id
)

SELECT
    start_date,
    end_date,
    COUNT(*) AS listing_count
FROM listing_ranges
GROUP BY
    start_date,
    end_date
ORDER BY
    start_date,
    end_date;

,start_date,end_date,listing_count
0,2026-06-30 00:00:00+00:00,2027-06-29 00:00:00+00:00,4305
1,2026-07-02 00:00:00+00:00,2027-07-01 00:00:00+00:00,644


## Profiling the columns

## availability

In [6]:
SELECT
    available,
    COUNT(*) AS calendar_dates,
    ROUND(
        100.0 * COUNT(*) / SUM(COUNT(*)) OVER (),
        2
    ) AS pct_calendar_dates
FROM 'detailed_calendar.csv'
GROUP BY available
ORDER BY calendar_dates DESC;

,available,calendar_dates,pct_calendar_dates
0,True,1021669,56.56
1,False,784716,43.44


In [8]:
SELECT
    COUNT(*) AS calendar_dates,
    SUM(CASE WHEN available = TRUE THEN 1 ELSE 0 END) AS available_dates,
    SUM(CASE WHEN available = FALSE THEN 1 ELSE 0 END) AS unavailable_dates
FROM 'detailed_calendar.csv';

,calendar_dates,available_dates,unavailable_dates
0,1806385,1021669.0,784716.0


In [9]:
WITH listing_availability AS (
    SELECT
        listing_id,
        COUNT(*) AS calendar_days,
        SUM(CASE WHEN available = TRUE THEN 1 ELSE 0 END)
            AS available_days
    FROM 'detailed_calendar.csv'
    GROUP BY listing_id
)

SELECT
    MIN(available_days) AS min_available_days,
    AVG(available_days) AS avg_available_days,
    MEDIAN(available_days) AS median_available_days,
    MAX(available_days) AS max_available_days
FROM listing_availability;

,min_available_days,avg_available_days,median_available_days,max_available_days
0,0.0,206.439483,236.0,365.0


In [10]:
WITH listing_availability AS (
    SELECT
        listing_id,
        SUM(CASE WHEN available = TRUE THEN 1 ELSE 0 END)
            AS available_days
    FROM 'detailed_calendar.csv'
    GROUP BY listing_id
)

SELECT
    CASE
        WHEN available_days = 0 THEN '0 days'
        WHEN available_days BETWEEN 1 AND 30 THEN '1-30 days'
        WHEN available_days BETWEEN 31 AND 90 THEN '31-90 days'
        WHEN available_days BETWEEN 91 AND 180 THEN '91-180 days'
        WHEN available_days BETWEEN 181 AND 270 THEN '181-270 days'
        WHEN available_days BETWEEN 271 AND 364 THEN '271-364 days'
        WHEN available_days = 365 THEN '365 days'
    END AS availability_bucket,
    COUNT(*) AS listing_count
FROM listing_availability
GROUP BY availability_bucket
ORDER BY MIN(available_days);

,availability_bucket,listing_count
0,0 days,379
1,1-30 days,280
2,31-90 days,594
3,91-180 days,764
4,181-270 days,909
5,271-364 days,1895
6,365 days,128


In [11]:
SELECT
    c.listing_id,
    COUNT(*) AS calendar_rows,
    MIN(c.date) AS first_date,
    MAX(c.date) AS last_date
FROM 'detailed_calendar.csv' AS c
LEFT JOIN 'detailed_listings.csv' AS l
    ON c.listing_id = l.id
WHERE l.id IS NULL
GROUP BY c.listing_id;

,listing_id,calendar_rows,first_date,last_date
0,1703706462421883312,365,2026-07-02 00:00:00+00:00,2027-07-01 00:00:00+00:00
1,1703706463365002102,365,2026-07-02 00:00:00+00:00,2027-07-01 00:00:00+00:00
2,1663911003117317942,365,2026-07-02 00:00:00+00:00,2027-07-01 00:00:00+00:00
3,1541416314294129738,365,2026-07-02 00:00:00+00:00,2027-07-01 00:00:00+00:00
4,1613930943834383250,365,2026-07-02 00:00:00+00:00,2027-07-01 00:00:00+00:00
5,1541416240915518927,365,2026-07-02 00:00:00+00:00,2027-07-01 00:00:00+00:00
6,1663911003765151859,365,2026-07-02 00:00:00+00:00,2027-07-01 00:00:00+00:00
7,1350903199350412273,365,2026-07-02 00:00:00+00:00,2027-07-01 00:00:00+00:00
8,1045288088562648513,365,2026-07-02 00:00:00+00:00,2027-07-01 00:00:00+00:00
9,1541416240576378877,365,2026-07-02 00:00:00+00:00,2027-07-01 00:00:00+00:00


In [12]:
WITH calendar_availability AS (
    SELECT
        listing_id,
        SUM(CASE WHEN available = TRUE THEN 1 ELSE 0 END)
            AS calendar_available_days
    FROM 'detailed_calendar.csv'
    GROUP BY listing_id
)

SELECT
    COUNT(*) AS matched_listings,

    SUM(
        CASE
            WHEN l.availability_365 = c.calendar_available_days
            THEN 1 ELSE 0
        END
    ) AS exact_matches,

    SUM(
        CASE
            WHEN l.availability_365 != c.calendar_available_days
            THEN 1 ELSE 0
        END
    ) AS mismatches,

    MAX(
        ABS(l.availability_365 - c.calendar_available_days)
    ) AS max_difference

FROM 'detailed_listings.csv' l
INNER JOIN calendar_availability c
    ON l.id = c.listing_id;

,matched_listings,exact_matches,mismatches,max_difference
0,4939,4938.0,1.0,207.0


In [14]:
WITH calendar_availability AS (
    SELECT
        listing_id,
        SUM(CASE WHEN available = TRUE THEN 1 ELSE 0 END)
            AS calendar_available_days
    FROM 'detailed_calendar.csv'
    GROUP BY listing_id
)

SELECT
    l.id,
    l.availability_30,
    l.availability_60,
    l.availability_90,
    l.availability_365,
    c.calendar_available_days,
    l.availability_365 - c.calendar_available_days
        AS difference
FROM 'detailed_listings.csv' l
INNER JOIN calendar_availability c
    ON l.id = c.listing_id
WHERE l.availability_365 != c.calendar_available_days;

,id,availability_30,availability_60,availability_90,availability_365,calendar_available_days,difference
0,1679140577476157015,0,0,0,0,207.0,-207.0


In [18]:
SELECT
    listing_id,
    MIN(date) AS start_date,
    MAX(date) AS end_date,
    COUNT(*) AS calendar_days,
    SUM(CASE WHEN available = TRUE THEN 1 ELSE 0 END)
        AS available_days,
    SUM(CASE WHEN available = FALSE THEN 1 ELSE 0 END)
        AS unavailable_days
FROM 'detailed_calendar.csv'
WHERE listing_id = TRIM('1679140577476157000')
GROUP BY listing_id;

,listing_id,start_date,end_date,calendar_days,available_days,unavailable_days


In [16]:
WITH calendar_availability AS (
    SELECT
        listing_id,
        SUM(CASE WHEN available = TRUE THEN 1 ELSE 0 END)
            AS calendar_available_days
    FROM 'detailed_calendar.csv'
    GROUP BY listing_id
),

mismatch AS (
    SELECT
        c.listing_id
    FROM calendar_availability c
    INNER JOIN 'detailed_listings.csv' l
        ON c.listing_id = l.id
    WHERE c.calendar_available_days != l.availability_365
)

SELECT
    c.listing_id,
    MIN(c.date) AS start_date,
    MAX(c.date) AS end_date,
    COUNT(*) AS calendar_days,
    SUM(CASE WHEN c.available = TRUE THEN 1 ELSE 0 END)
        AS available_days,
    SUM(CASE WHEN c.available = FALSE THEN 1 ELSE 0 END)
        AS unavailable_days
FROM 'detailed_calendar.csv' c
INNER JOIN mismatch m
    ON c.listing_id = m.listing_id
GROUP BY c.listing_id;

,listing_id,start_date,end_date,calendar_days,available_days,unavailable_days
0,1679140577476157015,2026-06-30 00:00:00+00:00,2027-06-29 00:00:00+00:00,365,207.0,158.0


In [19]:
WITH calendar_availability AS (
    SELECT
        listing_id,
        SUM(CASE WHEN available = TRUE THEN 1 ELSE 0 END)
            AS calendar_available_days
    FROM 'detailed_calendar.csv'
    GROUP BY listing_id
),

mismatch AS (
    SELECT
        c.listing_id
    FROM calendar_availability c
    INNER JOIN 'detailed_listings.csv' l
        ON c.listing_id = l.id
    WHERE c.calendar_available_days != l.availability_365
)

SELECT
    DATE_TRUNC('month', c.date) AS month,
    COUNT(*) AS calendar_days,
    SUM(CASE WHEN c.available = TRUE THEN 1 ELSE 0 END)
        AS available_days,
    SUM(CASE WHEN c.available = FALSE THEN 1 ELSE 0 END)
        AS unavailable_days
FROM 'detailed_calendar.csv' c
INNER JOIN mismatch m
    ON c.listing_id = m.listing_id
GROUP BY DATE_TRUNC('month', c.date)
ORDER BY month;

,month,calendar_days,available_days,unavailable_days
0,2026-06-01 00:00:00+00:00,1,0.0,1.0
1,2026-07-01 00:00:00+00:00,31,7.0,24.0
2,2026-08-01 00:00:00+00:00,31,22.0,9.0
3,2026-09-01 00:00:00+00:00,30,26.0,4.0
4,2026-10-01 00:00:00+00:00,31,31.0,0.0
5,2026-11-01 00:00:00+00:00,30,0.0,30.0
6,2026-12-01 00:00:00+00:00,31,0.0,31.0
7,2027-01-01 00:00:00+00:00,31,0.0,31.0
8,2027-02-01 00:00:00+00:00,28,0.0,28.0
9,2027-03-01 00:00:00+00:00,31,31.0,0.0


In [20]:
SELECT
    date,
    COUNT(*) AS listings_observed,
    SUM(CASE WHEN available = TRUE THEN 1 ELSE 0 END)
        AS available_listings,
    ROUND(
        100.0 *
        SUM(CASE WHEN available = TRUE THEN 1 ELSE 0 END)
        / COUNT(*),
        2
    ) AS availability_rate
FROM 'detailed_calendar.csv'
GROUP BY date
ORDER BY date;

,date,listings_observed,available_listings,availability_rate
0,2026-06-30 00:00:00+00:00,4305,683.0,15.87
1,2026-07-01 00:00:00+00:00,4305,807.0,18.75
2,2026-07-02 00:00:00+00:00,4949,769.0,15.54
3,2026-07-03 00:00:00+00:00,4949,710.0,14.35
4,2026-07-04 00:00:00+00:00,4949,703.0,14.20
...,...,...,...,...
362,2027-06-27 00:00:00+00:00,4949,2381.0,48.11
363,2027-06-28 00:00:00+00:00,4949,2384.0,48.17
364,2027-06-29 00:00:00+00:00,4949,2377.0,48.03
365,2027-06-30 00:00:00+00:00,644,99.0,15.37


In [21]:
SELECT
    DATE_TRUNC('month', date) AS month,
    COUNT(*) AS calendar_records,
    COUNT(DISTINCT listing_id) AS listings_represented,
    SUM(CASE WHEN available = TRUE THEN 1 ELSE 0 END)
        AS available_records,

    ROUND(
        100.0 *
        SUM(CASE WHEN available = TRUE THEN 1 ELSE 0 END)
        / COUNT(*),
        2
    ) AS availability_rate

FROM 'detailed_calendar.csv'

GROUP BY DATE_TRUNC('month', date)
ORDER BY month;

,month,calendar_records,listings_represented,available_records,availability_rate
0,2026-06-01 00:00:00+00:00,4305,4305,683.0,15.87
1,2026-07-01 00:00:00+00:00,152775,4949,47898.0,31.35
2,2026-08-01 00:00:00+00:00,153419,4949,79788.0,52.01
3,2026-09-01 00:00:00+00:00,148470,4949,91553.0,61.66
4,2026-10-01 00:00:00+00:00,153419,4949,95245.0,62.08
5,2026-11-01 00:00:00+00:00,148470,4949,98273.0,66.19
6,2026-12-01 00:00:00+00:00,153419,4949,102177.0,66.60
7,2027-01-01 00:00:00+00:00,153419,4949,95462.0,62.22
8,2027-02-01 00:00:00+00:00,138572,4949,87428.0,63.09
9,2027-03-01 00:00:00+00:00,153419,4949,94921.0,61.87


## minimum and maximum nights

In [3]:
SELECT 
	COUNT(*) AS calendar_records,
	SUM(CASE WHEN minimum_nights IS NULL THEN 1 ELSE 0 END) AS null_minimum_nights,
	MIN(minimum_nights) AS min_minimum_nights,
	MAX(minimum_nights) AS max_minimum_nights,
	COUNT(DISTINCT minimum_nights) AS distinct_minimum_nights
FROM 'detailed_calendar.csv';

,calendar_records,null_minimum_nights,min_minimum_nights,max_minimum_nights,distinct_minimum_nights
0,1806385,0.0,1,500,53


In [2]:
SELECT
    COUNT(*) AS listings,
    SUM(
        CASE
            WHEN distinct_minimum_nights > 1 THEN 1
            ELSE 0
        END
    ) AS listings_with_variable_minimums
FROM (
    SELECT
        listing_id,
        COUNT(DISTINCT minimum_nights)
            AS distinct_minimum_nights
    FROM 'detailed_calendar.csv'
    GROUP BY listing_id
);

,listings,listings_with_variable_minimums
0,4949,1213.0


In [6]:
SELECT
    COUNT(*) AS calendar_records,
    SUM(CASE WHEN maximum_nights IS NULL THEN 1 ELSE 0 END)
        AS null_maximum_nights,
    MIN(maximum_nights) AS min_maximum_nights,
    MAX(maximum_nights) AS max_maximum_nights,
    COUNT(DISTINCT maximum_nights)
        AS distinct_maximum_nights
FROM 'detailed_calendar.csv';

,calendar_records,null_maximum_nights,min_maximum_nights,max_maximum_nights,distinct_maximum_nights
0,1806385,0.0,1,1125,274


In [5]:
SELECT
    COUNT(*) AS listings,
    SUM(
        CASE
            WHEN distinct_maximum_nights > 1 THEN 1
            ELSE 0
        END
    ) AS listings_with_variable_maximums
FROM (
    SELECT
        listing_id,
        COUNT(DISTINCT maximum_nights)
            AS distinct_maximum_nights
    FROM 'detailed_calendar.csv'
    GROUP BY listing_id
);

,listings,listings_with_variable_maximums
0,4949,125.0


In [7]:
WITH calendar_comparison AS (
    SELECT
        l.id,
        MAX(
            CASE
                WHEN l.minimum_nights = c.minimum_nights
                THEN 1 ELSE 0
            END
        ) AS minimum_found_in_calendar,
        MAX(
            CASE
                WHEN l.maximum_nights = c.maximum_nights
                THEN 1 ELSE 0
            END
        ) AS maximum_found_in_calendar
    FROM 'detailed_listings.csv' l
    INNER JOIN 'detailed_calendar.csv' c
        ON l.id = c.listing_id
    GROUP BY l.id
)

SELECT
    COUNT(*) AS matched_listings,
    SUM(minimum_found_in_calendar)
        AS minimum_exact_matches,
    SUM(
        CASE
            WHEN minimum_found_in_calendar = 0 THEN 1
            ELSE 0
        END
    ) AS minimum_no_match,
    SUM(maximum_found_in_calendar)
        AS maximum_exact_matches,
    SUM(
        CASE
            WHEN maximum_found_in_calendar = 0 THEN 1
            ELSE 0
        END
    ) AS maximum_no_match
FROM calendar_comparison;

,matched_listings,minimum_exact_matches,minimum_no_match,maximum_exact_matches,maximum_no_match
0,4939,4938.0,1.0,4938.0,1.0


In [1]:
WITH calendar_summary AS (
    SELECT
        listing_id,

        MIN(minimum_nights) AS min_calendar_minimum,
        MAX(minimum_nights) AS max_calendar_minimum,

        MIN(maximum_nights) AS min_calendar_maximum,
        MAX(maximum_nights) AS max_calendar_maximum

    FROM 'detailed_calendar.csv'
    GROUP BY listing_id
),

comparison AS (
    SELECT
        l.id,
        l.minimum_nights AS listing_minimum,
        l.maximum_nights AS listing_maximum,

        c.min_calendar_minimum,
        c.max_calendar_minimum,
        c.min_calendar_maximum,
        c.max_calendar_maximum,

        MAX(
            CASE WHEN l.minimum_nights = cal.minimum_nights
                 THEN 1 ELSE 0 END
        ) AS minimum_found,

        MAX(
            CASE WHEN l.maximum_nights = cal.maximum_nights
                 THEN 1 ELSE 0 END
        ) AS maximum_found

    FROM 'detailed_listings.csv' l

    INNER JOIN calendar_summary c
        ON l.id = c.listing_id

    INNER JOIN 'detailed_calendar.csv' cal
        ON l.id = cal.listing_id

    GROUP BY
        l.id,
        l.minimum_nights,
        l.maximum_nights,
        c.min_calendar_minimum,
        c.max_calendar_minimum,
        c.min_calendar_maximum,
        c.max_calendar_maximum
)

SELECT *
FROM comparison
WHERE minimum_found = 0
   OR maximum_found = 0;

,id,listing_minimum,listing_maximum,min_calendar_minimum,max_calendar_minimum,min_calendar_maximum,max_calendar_maximum,minimum_found,maximum_found
0,1679140577476157015,<NA>,<NA>,1,1,365,365,0,0


In [2]:
/* Does any calendar record have minimum_nights > maximum_nights? */

SELECT 
	listing_id,
	date,
	minimum_nights,
	maximum_nights
FROM 'detailed_calendar.csv'
WHERE minimum_nights > maximum_nights
ORDER BY listing_id, date;

,listing_id,date,minimum_nights,maximum_nights
0,10927867,2026-07-02 00:00:00+00:00,29,1
1,10927867,2026-07-04 00:00:00+00:00,29,14
2,10927867,2026-07-05 00:00:00+00:00,29,14
3,10927867,2026-07-06 00:00:00+00:00,29,14
4,10927867,2026-07-07 00:00:00+00:00,29,14
...,...,...,...,...
3603,1462401546875554922,2027-06-25 00:00:00+00:00,29,14
3604,1462401546875554922,2027-06-26 00:00:00+00:00,29,14
3605,1462401546875554922,2027-06-27 00:00:00+00:00,29,14
3606,1462401546875554922,2027-06-28 00:00:00+00:00,29,14


In [3]:
SELECT
    COUNT(*) AS invalid_calendar_records,
    COUNT(DISTINCT listing_id) AS affected_listings
FROM 'detailed_calendar.csv'
WHERE minimum_nights > maximum_nights;

,invalid_calendar_records,affected_listings
0,3608,14


In [4]:
SELECT
    listing_id,
    COUNT(*) AS calendar_days,
    SUM(
        CASE
            WHEN minimum_nights > maximum_nights
            THEN 1 ELSE 0
        END
    ) AS minimum_exceeds_maximum_days,
    MIN(minimum_nights) AS min_minimum_nights,
    MAX(minimum_nights) AS max_minimum_nights,
    MIN(maximum_nights) AS min_maximum_nights,
    MAX(maximum_nights) AS max_maximum_nights
FROM 'detailed_calendar.csv'
GROUP BY listing_id
HAVING SUM(
    CASE
        WHEN minimum_nights > maximum_nights
        THEN 1 ELSE 0
    END
) > 0
ORDER BY minimum_exceeds_maximum_days DESC;

,listing_id,calendar_days,minimum_exceeds_maximum_days,min_minimum_nights,max_minimum_nights,min_maximum_nights,max_maximum_nights
0,1462401546875554922,365,365.0,29,29,14,14
1,31540416,365,364.0,29,29,1,365
2,10927867,365,362.0,29,29,1,365
3,10927919,365,361.0,29,29,1,365
4,28545094,365,361.0,29,29,1,365
5,28545121,365,360.0,29,29,1,365
6,21976300,365,329.0,29,29,1,1125
7,21992685,365,290.0,29,29,1,1125
8,22727793,365,272.0,29,29,1,1125
9,22747038,365,271.0,29,29,1,1125


## detailed_reviews

In [1]:
SELECT 
	COUNT(*) AS total_rows,
	COUNT(DISTINCT id) AS reviews,
	COUNT(DISTINCT listing_id) AS listings,
	COUNT(DISTINCT reviewer_id) AS reviewers,
	MIN(date) AS min_date,
	MAX(date) AS max_date
FROM 'detailed_reviews.csv';

,total_rows,reviews,listings,reviewers,min_date,max_date
0,354741,354741,4309,316556,2008-08-28 00:00:00+00:00,2026-06-30 00:00:00+00:00


In [4]:
SELECT 
	COUNT(*) AS total_rows,
	SUM(CASE WHEN id IS NULL THEN 1 ELSE 0 END) AS null_review_ids
FROM 'detailed_reviews.csv';

,total_rows,null_review_ids
0,354741,0.0


In [5]:
SELECT 
	id,
	COUNT(*) AS row_count
FROM 'detailed_reviews.csv'
GROUP BY id
HAVING COUNT(*) > 1
ORDER BY row_count DESC;

,id,row_count


In [8]:
SELECT 
	COUNT(*) AS unmatched_reviews,
	COUNT(DISTINCT r.listing_id) AS unmatched_listing_ids
FROM 'detailed_reviews.csv' AS r
LEFT JOIN 'detailed_listings.csv' AS l
	ON r.listing_id = l.id
WHERE l.id IS NULL;

,unmatched_reviews,unmatched_listing_ids
0,95,8


In [9]:
SELECT 
	COUNT(DISTINCT l.id) AS listings_with_no_reviews
FROM 'detailed_listings.csv' AS l
LEFT JOIN 'detailed_reviews.csv' AS r
	ON l.id = r.listing_id
WHERE r.listing_id IS NULL;

,listings_with_no_reviews
0,638


In [10]:
WITH review_counts AS (
	SELECT 
		l.id AS listing_id,
		l.number_of_reviews AS reported_reviews,
		COUNT(r.id) AS actual_review_records
	FROM 'detailed_listings.csv' AS l
	LEFT JOIN 'detailed_reviews.csv' AS r
		ON l.id = r.listing_id
	GROUP BY l.id, l.number_of_reviews
)

SELECT 
	COUNT(*) AS listings,
	SUM(CASE WHEN reported_reviews = actual_review_records THEN 1 ELSE 0 END) AS exact_matches,
	SUM(CASE WHEN reported_reviews != actual_review_records THEN 1 ELSE 0 END) AS mismatches
FROM review_counts;

,listings,exact_matches,mismatches
0,4939,4939.0,0.0


In [11]:
WITH review_counts AS (
    SELECT
        l.id AS listing_id,
        l.number_of_reviews AS reported_reviews,
        COUNT(r.id) AS actual_review_records
    FROM 'detailed_listings.csv' l
    LEFT JOIN 'detailed_reviews.csv' r
        ON l.id = r.listing_id
    GROUP BY
        l.id,
        l.number_of_reviews
)

SELECT
    listing_id,
    reported_reviews,
    actual_review_records,
    reported_reviews - actual_review_records AS difference
FROM review_counts
WHERE reported_reviews != actual_review_records
ORDER BY ABS(difference) DESC;

,listing_id,reported_reviews,actual_review_records,difference


In [13]:
SELECT 
	COUNT(*) AS total_reviews,
	SUM(CASE WHEN reviewer_id IS NULL THEN 1 ELSE 0 END) AS null_reviewer_ids,
	SUM(CASE WHEN reviewer_name IS NULL THEN 1 ELSE 0 END) AS null_reviewer_names,
	SUM(CASE WHEN reviewer_id IS NULL AND reviewer_name IS NOT NULL THEN 1 ELSE 0 END) AS id_missing_name_present,
	SUM(CASE WHEN reviewer_id IS NOT NULL AND reviewer_name IS NULL THEN 1 ELSE 0 END) AS id_present_name_missing
FROM 'detailed_reviews.csv';

,total_reviews,null_reviewer_ids,null_reviewer_names,id_missing_name_present,id_present_name_missing
0,354741,0.0,0.0,0.0,0.0


In [4]:
WITH reviewer AS (
	SELECT 
		reviewer_id,
		COUNT(*) AS review_count
	FROM 'detailed_reviews.csv'
	GROUP BY reviewer_id
)

SELECT 
	review_count,
	COUNT(*) AS reviewer_count
FROM reviewer
GROUP BY review_count
ORDER BY review_count;

,review_count,reviewer_count
0,1,290444
1,2,19757
2,3,3870
3,4,1306
4,5,543
5,6,270
6,7,114
7,8,88
8,9,48
9,10,22


In [7]:
SELECT 
	reviewer_id,
	COUNT(*) AS review_count,
	COUNT(DISTINCT listing_id) AS distinct_listings
FROM 'detailed_reviews.csv'
GROUP BY reviewer_id
HAVING COUNT(*) > 1
ORDER BY review_count DESC;

,reviewer_id,review_count,distinct_listings
0,261185337,92,7
1,73366630,69,10
2,419669366,31,4
3,906007,29,27
4,247494246,27,9
...,...,...,...
26107,434449310,2,1
26108,546406355,2,2
26109,117170568,2,2
26110,63229495,2,1


In [16]:
WITH reviewer AS (
    SELECT
        reviewer_id,
        COUNT(*) AS review_count,
        COUNT(DISTINCT listing_id) AS distinct_listings
    FROM 'detailed_reviews.csv'
    GROUP BY reviewer_id
    HAVING COUNT(*) > 1
)

SELECT
    reviewer_id,
    review_count,
    distinct_listings
FROM reviewer
ORDER BY distinct_listings DESC
LIMIT 1;

,reviewer_id,review_count,distinct_listings
0,906007,29,27


In [11]:
WITH reviewer AS (
	SELECT 
		reviewer_id,
		COUNT(*) AS review_count,
		COUNT(DISTINCT listing_id) AS distinct_listings
	FROM 'detailed_reviews.csv'
	GROUP BY reviewer_id
	HAVING COUNT(*) > 1
)

SELECT 
	CASE 
		WHEN distinct_listings = 1 THEN 'One listing'
		WHEN distinct_listings BETWEEN 2 AND 10 THEN '2-10 listings'
		WHEN distinct_listings > 10 THEN 'Over 10 listings'
	END AS reviewer_category,
	COUNT(*) AS reviewer_count
FROM reviewer
GROUP BY reviewer_category
ORDER BY reviewer_count DESC;

,reviewer_category,reviewer_count
0,2-10 listings,23315
1,One listing,2763
2,Over 10 listings,34


In [14]:
SELECT 
	reviewer_id,
	listing_id,
	COUNT(*) AS review_count,
	COUNT(DISTINCT date) AS distinct_dates
FROM 'detailed_reviews.csv'
GROUP BY reviewer_id, listing_id
HAVING COUNT(*) > 1
ORDER BY review_count DESC;

,reviewer_id,listing_id,review_count,distinct_dates
0,73366630,42095197,45,45
1,261185337,9445553,41,41
2,261185337,643647236472324580,26,26
3,419669366,7052545,18,18
4,10553986,32550905,15,15
...,...,...,...,...
4927,21525999,54306154,2,2
4928,32857861,53865541,2,2
4929,495556015,535408542916708813,2,2
4930,345714027,556464390900551837,2,2


In [17]:
SELECT 
	reviewer_id,
	listing_id,
	date,
	COUNT(*) AS review_count
FROM 'detailed_reviews.csv'
GROUP BY reviewer_id, listing_id, date
HAVING COUNT(*) > 1 
ORDER BY review_count DESC;

,reviewer_id,listing_id,date,review_count
0,328769406,52691368,2023-06-23 00:00:00+00:00,2
1,267191199,970410863055824751,2024-08-23 00:00:00+00:00,2
2,5762349,10927867,2023-02-19 00:00:00+00:00,2
3,178454002,44334732,2023-08-28 00:00:00+00:00,2
4,3434221,44334753,2023-08-16 00:00:00+00:00,2
5,36533915,13751109,2016-09-26 00:00:00+00:00,2
6,296449294,31540416,2022-03-31 00:00:00+00:00,2


In [20]:
WITH duplicate_candidates AS (
    SELECT
        reviewer_id,
        listing_id,
        date
    FROM 'detailed_reviews.csv'
    GROUP BY
        reviewer_id,
        listing_id,
        date
    HAVING COUNT(*) > 1
)

SELECT
    r.*
FROM 'detailed_reviews.csv' r
INNER JOIN duplicate_candidates d
    ON r.reviewer_id = d.reviewer_id
    AND r.listing_id = d.listing_id
    AND r.date = d.date
ORDER BY
    r.reviewer_id,
    r.listing_id,
    r.date;

,listing_id,id,date,reviewer_id,reviewer_name,comments
0,44334753,959512584849136659,2023-08-16 00:00:00+00:00,3434221,Margo Alissan,Loved everything about our stay. Amazing locat...
1,44334753,959520046296540670,2023-08-16 00:00:00+00:00,3434221,Margo Alissan,"Beautiful modern building, amazing location cl..."
2,10927867,830530302915565606,2023-02-19 00:00:00+00:00,5762349,Lacey,Hip place for young people!
3,10927867,830543710250017500,2023-02-19 00:00:00+00:00,5762349,Lacey,Cool place!
4,13751109,104291828,2016-09-26 00:00:00+00:00,36533915,Deborah,"Perfect apartment, excellent location and comm..."
5,13751109,104452922,2016-09-26 00:00:00+00:00,36533915,Deborah,I found that I needed to stay another evening ...
6,44334732,968171811162973863,2023-08-28 00:00:00+00:00,178454002,Carl,Nice stay
7,44334732,968255666892411324,2023-08-28 00:00:00+00:00,178454002,Carl,Nice stay
8,970410863055824751,1229799387967198469,2024-08-23 00:00:00+00:00,267191199,Richard Walter,⭐️⭐️⭐️⭐️⭐️From the beginning Ali was very pati...
9,970410863055824751,1229852421290273857,2024-08-23 00:00:00+00:00,267191199,Richard Walter,A beautiful home to rest my head for a few nig...


### Content and temporal behavior of reviews

In [9]:
WITH review_years AS (
	SELECT 
		DATE_TRUNC('year', date) AS year,
		COUNT(*) AS reviews,
		COUNT(DISTINCT reviewer_id) AS reviewers,
		COUNT(DISTINCT listing_id) AS listings
	FROM 'detailed_reviews.csv'
	GROUP BY year
)

SELECT *,
	ROUND(reviews * 1.0 / reviewers, 2) AS reviews_per_reviewers,
	ROUND(reviews * 1.0 / listings, 2) AS reviews_per_reviewed_listing
FROM review_years
ORDER BY year;

,year,reviews,reviewers,listings,reviews_per_reviewers,reviews_per_reviewed_listing
0,2008-01-01 00:00:00+00:00,3,3,1,1.00,3.00
1,2009-01-01 00:00:00+00:00,19,17,3,1.12,6.33
2,2010-01-01 00:00:00+00:00,36,36,4,1.00,9.00
3,2011-01-01 00:00:00+00:00,82,82,6,1.00,13.67
4,2012-01-01 00:00:00+00:00,137,136,10,1.01,13.70
5,2013-01-01 00:00:00+00:00,290,285,39,1.02,7.44
6,2014-01-01 00:00:00+00:00,757,745,57,1.02,13.28
7,2015-01-01 00:00:00+00:00,2146,2114,141,1.02,15.22
8,2016-01-01 00:00:00+00:00,5957,5782,272,1.03,21.90
9,2017-01-01 00:00:00+00:00,11510,11264,411,1.02,28.00


In [5]:
/* Lifetime review-count distribution among listings represented in detailed_reviews.csv. */

WITH listing_reviews AS (
	SELECT 
		listing_id,
		COUNT(*) AS reviews
	FROM 'detailed_reviews.csv'
	GROUP BY listing_id
),

listing_category AS (
	SELECT 
		CASE 
			WHEN reviews < 100 THEN 'Under 100 Reviews'
			WHEN reviews BETWEEN 100 AND 499 THEN '100-499 Reviews'
			WHEN reviews BETWEEN 500 AND 999 THEN '500-999 Reviews'
			WHEN reviews >= 1000 THEN 'Over 1000 Reviews'
		END AS listing_reviews_category,
		COUNT(*) AS listings
	FROM listing_reviews
	GROUP BY listing_reviews_category
),

total AS (
	SELECT 
		COUNT(DISTINCT listing_id) AS total_listings
	FROM 'detailed_reviews.csv'
)

SELECT listing_reviews_category, listings,
	ROUND(listings/total_listings * 100, 2) AS percent_of_total_reviewed_listings
FROM listing_category AS l
CROSS JOIN total AS t
ORDER BY listings DESC;

,listing_reviews_category,listings,percent_of_total_reviewed_listings
0,Under 100 Reviews,3221,74.75
1,100-499 Reviews,998,23.16
2,500-999 Reviews,85,1.97
3,Over 1000 Reviews,5,0.12


In [31]:
WITH review_dates AS (
	SELECT 
		listing_id,
		MIN(date) AS earliest_review,
		MAX(date) AS latest_review
	FROM 'detailed_reviews.csv'
	GROUP BY listing_id
)

SELECT 
	l.id AS listing_id,
	r.earliest_review,
	l.first_review,
	r.latest_review,
	l.last_review
FROM 'detailed_listings.csv' AS l
INNER JOIN review_dates AS r
	ON l.id = r.listing_id
WHERE 
	r.earliest_review != l.first_review
	OR r.latest_review != l.last_review
ORDER BY listing_id;

,listing_id,earliest_review,first_review,latest_review,last_review


In [10]:
SELECT 
	r.listing_id,
	l.first_review,
	l.last_review
FROM 'detailed_listings.csv' AS l 
LEFT JOIN 'detailed_reviews.csv' AS r
	ON l.id = r.listing_id
WHERE r.listing_id IS NULL 
	AND (l.first_review IS NOT NULL 
	OR l.last_review IS NOT NULL);

,listing_id,first_review,last_review


## neighbourhoods.csv

In [1]:
SELECT *
FROM 'neighbourhoods.csv';

,neighbourhood_group,neighbourhood
0,None,Athmar Park
1,None,Auraria
2,None,Baker
3,None,Barnum
4,None,Barnum West
...,...,...
73,None,West Colfax
74,None,West Highland
75,None,Westwood
76,None,Whittier


In [2]:
SELECT 
	COUNT(*) AS rows,
	COUNT(CASE WHEN neighbourhood IS NULL THEN 1 END) AS null_neighbourhood_rows,
	COUNT(CASE WHEN neighbourhood_group IS NULL THEN 1 END) AS null_neighbourhood_group_rows,
	COUNT(DISTINCT neighbourhood) AS distinct_neighbourhoods,
FROM 'neighbourhoods.csv';

,rows,null_neighbourhood_rows,null_neighbourhood_group_rows,distinct_neighbourhoods
0,78,0,78,78


In [3]:
SELECT
    neighbourhood,
    COUNT(*) AS row_count
FROM 'neighbourhoods.csv'
GROUP BY neighbourhood
HAVING COUNT(*) > 1;

,neighbourhood,row_count


In [5]:
SELECT DISTINCT
	l.neighbourhood_cleansed
FROM 'detailed_listings.csv' AS l
LEFT JOIN 'neighbourhoods.csv' AS n
	ON l.neighbourhood_cleansed = n.neighbourhood
WHERE n.neighbourhood IS NULL;

,neighbourhood_cleansed


In [6]:
SELECT DISTINCT
    n.neighbourhood
FROM 'neighbourhoods.csv' n
LEFT JOIN 'detailed_listings.csv' l
    ON n.neighbourhood = l.neighbourhood_cleansed
WHERE l.neighbourhood_cleansed IS NULL;

,neighbourhood
0,Sun Valley
